In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/NLP and Gen-AI Batches/19 Apr 25 - Velocity NLP and Gen-AI Batch/Extra Notes/Datasets/complaints.csv')

In [ ]:
# /content/drive/MyDrive/NLP and Gen-AI Batches/19 Apr 25 - Velocity NLP and Gen-AI Batch/Extra Notes/Datasets/complaints.csv

In [4]:
## Drop unnecessary columns
df = df[["Product", "Consumer complaint narrative"]]

In [5]:
## Drop null values :
df.dropna(inplace=True)

In [6]:
class_dict = {
    'Vehicle loan or lease' : 'Loan',
    'Credit reporting, credit repair services, or other personal consumer reports' : 'Credit_Report',
    'Credit card or prepaid card' : "Card",
    'Money transfer, virtual currency, or money service' : 'Services',
    'Mortgage' : "Loan",
    'Payday loan, title loan, or personal loan' : 'Loan',
    'Debt collection' : 'Others',
    'Checking or savings account' : 'Others',
    'Student loan' : 'Loan',
    'Consumer Loan' : 'Loan',
    'Money transfers' : 'Services',
    'Credit card' : 'Card',
    'Bank account or service' : 'Services',
    'Credit reporting' : 'Credit_Report',
    'Prepaid card' : 'Card',
    'Payday loan'  :  'Loan',
    'Other financial service' : 'Services',
    'Virtual currency' : 'Services'
}

In [7]:
df.replace({"Product" : class_dict}, inplace=True)

In [8]:
import re

In [9]:
def text_preprocessing(complaint):

  ##Lower Case
  complaint = complaint.lower()

  ## Remove non-alpha
  complaint = re.sub(r'[^a-z. ]', '', complaint)

  ## Remove masking (XXXX)
  complaint = re.sub(r'[x]{2,}', '', complaint)

  return complaint


In [10]:
df['Consumer complaint narrative'] = df['Consumer complaint narrative'].apply(text_preprocessing)

In [11]:
df["Product"].value_counts()

,count
Product,
Credit_Report,348053
Others,194770
Loan,150205
Card,81042
Services,35273


In [ ]:
##Data is imbalance :

In [12]:
df.shape

(809343, 2)

In [ ]:
>> BERT (any pretrained model) will give bettwe results on less data
>> for finetuning on more data more computational cost required.

##Undersampling :

In [13]:
sampled_df = pd.DataFrame()

In [14]:
df["Product"].unique()

array(['Loan', 'Credit_Report', 'Card', 'Services', 'Others'],
      dtype=object)

In [15]:
for department in df["Product"].unique():
  temp_df = df[df["Product"] == department]
  sampled_df = pd.concat([sampled_df, temp_df.sample(n=1000)])

In [ ]:
##We can choose 35k per class,
# but becuase of computational constraints we choose 1k

In [16]:
sampled_df.shape

(5000, 2)

In [17]:
sampled_df["Product"].value_counts()

,count
Product,
Loan,1000
Credit_Report,1000
Card,1000
Services,1000
Others,1000


In [18]:
df.head()

,Product,Consumer complaint narrative
1,Loan,i contacted ally on friday after falling behi...
7,Credit_Report,hello this complaint is against the three cred...
8,Credit_Report,i am a victim of identity theft currently hav...
10,Credit_Report,two accounts are still on my credit history af...
13,Credit_Report,receiving daily telephone call s from ide...


In [19]:
classes = {
    'Loan' : 0,
    'Credit_Report' : 1,
    'Card' : 2,
    'Services' : 3,
    'Others' : 4
}

In [20]:
sampled_df.replace({"Product" : classes}, inplace=True)

/tmp/ipython-input-20-352164751.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sampled_df.replace({"Product" : classes}, inplace=True)


In [21]:
sampled_df['Product'].value_counts()

,count
Product,
0,1000
1,1000
2,1000
3,1000
4,1000


In [26]:
# sampled_df.to_csv('sampled_complaints.csv', index=False)

###Dataset :

In [ ]:
## all types of data must be converted to huggingface dataset

In [ ]:
!pip install datasets

In [ ]:
##Downloading dataset from huggingface hub
from datasets import load_dataset

data = load_dataset("fka/awesome-chatgpt-prompts")

In [ ]:
# from datasets import load_dataset

# ds = load_dataset("fka/awesome-chatgpt-prompts")

In [24]:
##Convert to dataset from DataFrame
from datasets import Dataset

ds = Dataset.from_pandas(sampled_df)

In [ ]:
# type(ds)

In [ ]:
##Load csv as a dataset
from datasets import load_dataset

dataset = load_dataset("csv",
                       data_files="data/train.csv")

In [25]:
ds

Dataset({
    features: ['Product', 'Consumer complaint narrative', '__index_level_0__'],
    num_rows: 5000
})

In [ ]:
## Target column must be named as "label",
## other names such as 'Product' are not accepted

In [ ]:
# Train :
# Transfer Learning :
# Finetuning :

In [26]:
train_test = ds.train_test_split(test_size=0.2)

In [27]:
train_test["train"]

Dataset({
    features: ['Product', 'Consumer complaint narrative', '__index_level_0__'],
    num_rows: 4000
})

In [28]:
train_test["test"]

Dataset({
    features: ['Product', 'Consumer complaint narrative', '__index_level_0__'],
    num_rows: 1000
})

###Model & Tokenizer :

In [29]:
checkpoint = "bert-base-cased"

In [30]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [31]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,
                                                           num_labels=5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Some weights of BertForSequenceClassification were not initialized
# from the model checkpoint at bert-base-cased and are newly initialized:
#  ['classifier.bias', 'classifier.weight']
# You should probably TRAIN this model on a down-stream task to be able
# to use it for predictions and inference.

In [32]:
tokenizer("this is complaint")

{'input_ids': [101, 1142, 1110, 12522, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [33]:
def tokenize(batch):
  return tokenizer(batch["Consumer complaint narrative"],
                   padding=True,
                   truncation=True,
                  #  max_length=500
                   )

In [37]:
tokenized_data = train_test.map(tokenize)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [35]:
# tokenized_data

In [36]:
from transformers import Trainer, TrainingArguments

In [38]:
training_args = TrainingArguments(
    output_dir="test_trainer",      ##path to to save model checkpoints
    # evaluation_strategy="epoch" ,    ##after every epoch
    save_strategy="epoch",       ##When to save checkpoints
    num_train_epochs=1,   ##How many epochs to train
    report_to="none",


    )

In [39]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [40]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
  acc = accuracy_score(labels, predictions)
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

In [41]:
tokenized_data = tokenized_data.rename_column("Product", "labels")

In [42]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    compute_metrics=compute_metrics
)

/tmp/ipython-input-42-1300304027.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [43]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['labels', 'Consumer complaint narrative', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['labels', 'Consumer complaint narrative', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [44]:
trainer.train()

Step,Training Loss
500,0.862900


TrainOutput(global_step=500, training_loss=0.8629442749023437, metrics={'train_runtime': 359.723, 'train_samples_per_second': 11.12, 'train_steps_per_second': 1.39, 'total_flos': 904674175768800.0, 'train_loss': 0.8629442749023437, 'epoch': 1.0})

In [45]:
new_model = AutoModelForSequenceClassification.from_pretrained("/content/test_trainer/checkpoint-500")

In [46]:
tokenizer = AutoTokenizer.from_pretrained("/content/test_trainer/checkpoint-500")

In [47]:
from transformers import pipeline

classifier = pipeline("text-classification",
                      model=new_model,
                      tokenizer=tokenizer)

Device set to use cuda:0


In [49]:
new_model.config.label2id

{'LABEL_0': 0, 'LABEL_1': 1, 'LABEL_2': 2, 'LABEL_3': 3, 'LABEL_4': 4}

In [50]:
class_map = {
   'LABEL_0' : 'Loan',
    'LABEL_1' : 'Credit_Report',
    'LABEL_2' : 'Card',
    'LABEL_3' : 'Services',
    'LABEL_4' : 'Others'
}

In [53]:
output = classifier("my credit card is not working")
output

[{'label': 'LABEL_2', 'score': 0.7843906879425049}]

In [55]:
class_map[output[0]["label"]]

'Card'